<a href="https://colab.research.google.com/github/henryparfait/kinyarwanda-news-classifier/blob/main/notebooks/06_deploy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Deployment: Hugging Face Hub (model) + Streamlit Community Cloud (web app)

**Why this setup:** Hugging Face now requires a paid plan to host Gradio/Docker apps on its free hardware, so the **model** is hosted on the Hugging Face Hub (free), and the **web app** runs on **Streamlit Community Cloud** (free, deployed directly from the GitHub repo).

**How the app connects to the model:** the app starts → `predictor.py` downloads the model + tokenizer from the Hub (once, then cached) → the user pastes text → tokenise (truncate to max length) → model → softmax → top-5 categories + confidence + warnings.

This notebook:
1. Uploads the best model to the Hub with a model card (re-running is safe, since unchanged files are skipped)
2. Writes the app files: `predictor.py` (model logic), `streamlit_app.py` (interface), `requirements.txt`, `model_info.json`, `examples.json`



## Step 1: Log in to Hugging Face


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, shutil
import pandas as pd
from huggingface_hub import notebook_login, HfApi
notebook_login()

Mounted at /content/drive


## Step 2: Names and links.

In [ ]:
PROJECT_DIR = '/content/drive/MyDrive/kinyarwanda-news-classifier'
DATA_DIR, RESULTS_DIR, MODEL_DIR = f'{PROJECT_DIR}/data', f'{PROJECT_DIR}/results', f'{PROJECT_DIR}/models'
GITHUB_URL = 'https://github.com/henryparfait/kinyarwanda-news-classifier'   # <-- edit this

api = HfApi()
USER = api.whoami()['name']
MODEL_REPO = f'{USER}/kinnews-topic-classifier'

run_info = json.load(open(f'{MODEL_DIR}/best_transformer/run_info.json'))
log = pd.read_csv(f'{RESULTS_DIR}/experiment_log.csv')
best_row = log[log['exp_id'] == run_info['exp_id']].iloc[0]
test_size = len(pd.read_csv(f'{DATA_DIR}/test_dedup.csv'))
print('Model repo:', MODEL_REPO); print(run_info)
print(best_row[['exp_id', 'test_accuracy', 'test_macro_f1']].to_dict())

Model repo: henryparfait/kinnews-topic-classifier
{'exp_id': 'T3', 'model': 'castorini/afriberta_base', 'max_len': 512}
{'exp_id': 'T3', 'test_accuracy': 0.7546, 'test_macro_f1': 0.581}


## Step 3: Model card + upload the model to the Hub

In [ ]:
UPLOAD_DIR = '/content/model_upload'
if os.path.exists(UPLOAD_DIR): shutil.rmtree(UPLOAD_DIR)
shutil.copytree(f'{MODEL_DIR}/best_transformer', UPLOAD_DIR)

card = f'''---
language: rw
license: mit
base_model: {run_info['model']}
pipeline_tag: text-classification
tags: [kinyarwanda, news, topic-classification, african-nlp]
---

# Kinyarwanda News Topic Classifier

`{run_info['model']}` fine-tuned to classify Kinyarwanda news articles into 14 topics
(KINNEWS categories). Built as an individual NLP capstone project.

## Training data
KINNEWS (Niyongabo, Hong, Kurtz and Cao, 2020; MIT license). Exact duplicates (over half of the raw training rows)
and 156 articles with conflicting labels were removed; 10% of the remaining training data was held out for validation.

## Training
Max length {run_info['max_len']} tokens, AdamW (lr 3e-5, linear warm-up and decay), class-weighted cross-entropy,
early stopping on validation macro-F1.

## Results (deduplicated KINNEWS test set, {test_size} articles)
| Metric | Score |
|---|---|
| Accuracy | {best_row['test_accuracy']:.4f} |
| Macro-F1 | {best_row['test_macro_f1']:.4f} |
| Weighted-F1 | {best_row['test_weighted_f1']:.4f} |

## Intended use and limitations
For research and demonstration on Kinyarwanda news text. Rare categories (tourism, fashion, environment, education) are
much less reliable; class distributions differ strongly between the KINNEWS train and test splits; the model can partly rely
on source names and bylines; it was not trained on social media, non-news or non-Kinyarwanda text.

Code and full report: {GITHUB_URL}
'''
open(f'{UPLOAD_DIR}/README.md', 'w').write(card)

api.create_repo(MODEL_REPO, repo_type='model', exist_ok=True)
api.upload_folder(folder_path=UPLOAD_DIR, repo_id=MODEL_REPO, repo_type='model',
                  commit_message='Upload fine-tuned Kinyarwanda news classifier')
print('Model on the Hub: https://huggingface.co/' + MODEL_REPO)

No files have been modified since last commit. Skipping to prevent empty commit.


Model on the Hub: https://huggingface.co/henryparfait/kinnews-topic-classifier


## Step 4: App data files (`model_info.json`, `examples.json`)
The examples include confident correct predictions, two known model errors and one English (out-of-scope) input, so the demo shows both successful and unsuccessful cases.

In [ ]:
APP_DIR = '/content/app'
if os.path.exists(APP_DIR): shutil.rmtree(APP_DIR)
os.makedirs(APP_DIR)

json.dump({'model_repo': MODEL_REPO, 'max_len': int(run_info['max_len']), 'base_model': run_info['model'],
           'exp_id': run_info['exp_id'], 'test_accuracy': float(best_row['test_accuracy']),
           'test_macro_f1': float(best_row['test_macro_f1']), 'test_size': test_size, 'github_url': GITHUB_URL},
          open(f'{APP_DIR}/model_info.json', 'w'), indent=2)

tp = pd.read_csv(f'{RESULTS_DIR}/best_transformer_test_predictions.csv')
def excerpt(t, n=120):
    w = t.split()
    return ' '.join(w[:n]) + (' …' if len(w) > n else '')

examples = []
for cat in ['sport', 'health', 'religion', 'economy']:
    ok = tp[(tp['true'] == cat) & (tp['pred'] == cat)].sort_values('confidence', ascending=False)
    if len(ok):
        examples.append({'text': excerpt(ok.iloc[0]['text']), 'description': f'{cat} article'})
wrong = tp[tp['true'] != tp['pred']].sort_values('confidence', ascending=False)
for _, r in wrong.head(2).iterrows():
    examples.append({'text': excerpt(r['text']),
                     'description': f"labelled {r['true']} (model error on full article: {r['pred']})"})
examples.append({'text': 'The central bank raised its key interest rate on Tuesday, citing rising inflation '
                         'and a weaker currency, and warned that prices could keep climbing this year.',
                 'description': 'English text (out of scope)'})
json.dump(examples, open(f'{APP_DIR}/examples.json', 'w', encoding='utf-8'), indent=2, ensure_ascii=False)
for e in examples: print('-', e['description'])

- sport article
- health article
- religion article
- economy article
- labelled entertainment (model error on full article: sport)
- labelled entertainment (model error on full article: sport)
- English text (out of scope)


## Step 5: `predictor.py`, the model logic (loading + prediction)

In [ ]:
%%writefile /content/app/predictor.py
"""Model loading and prediction for the Kinyarwanda News Topic Classifier.

This module is the bridge between the web interface and the ML model:
the fine-tuned Transformer is downloaded from the Hugging Face Hub, each input
text is tokenised (truncated to MAX_LEN tokens), passed through the model, and
softmax probabilities over the 14 KINNEWS categories are returned, ranked.
"""
import json
import os

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

HERE = os.path.dirname(os.path.abspath(__file__))
with open(os.path.join(HERE, "model_info.json"), encoding="utf-8") as f:
    INFO = json.load(f)
MODEL_ID = INFO["model_repo"]
MAX_LEN = int(INFO["max_len"])


def load_model():
    try:
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    except Exception:
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID)
    model.eval()
    id2label = {int(k): v for k, v in model.config.id2label.items()}
    return tokenizer, model, id2label


def classify(text, tokenizer, model, id2label):
    """Return ([(category, probability), ...] sorted high to low, number of tokens in the text)."""
    enc = tokenizer(text, truncation=True, max_length=MAX_LEN, return_tensors="pt")
    n_tokens = len(tokenizer(text, add_special_tokens=False)["input_ids"])
    with torch.no_grad():
        probs = torch.softmax(model(**enc).logits, dim=-1)[0]
    ranked = sorted(((id2label[i], float(p)) for i, p in enumerate(probs)), key=lambda x: x[1], reverse=True)
    return ranked, n_tokens

Writing /content/app/predictor.py


## Step 6: `streamlit_app.py`, the web interface

In [ ]:
%%writefile /content/app/streamlit_app.py
"""Kinyarwanda News Topic Classifier: Streamlit web interface.

Model loading and prediction live in predictor.py.
"""
import json
import os

import streamlit as st

from predictor import INFO, MAX_LEN, classify, load_model

HERE = os.path.dirname(os.path.abspath(__file__))
with open(os.path.join(HERE, "examples.json"), encoding="utf-8") as f:
    EXAMPLES = json.load(f)
LOW_CONFIDENCE = 0.5
NO_EXAMPLE = "Type or paste your own article"
EXAMPLE_NAMES = [f"{i + 1}. {e['description']}" for i, e in enumerate(EXAMPLES)]

st.set_page_config(page_title="Kinyarwanda News Topic Classifier", page_icon="📰", layout="centered")


@st.cache_resource(show_spinner="Loading the model… (the first load can take about a minute)")
def get_model():
    return load_model()


tokenizer, model, id2label = get_model()
categories = ", ".join(id2label[i] for i in sorted(id2label))

st.title("📰 Kinyarwanda News Topic Classifier")
st.write("Paste a **Kinyarwanda news article** (title + text) and the model predicts its topic "
         f"among 14 categories: {categories}.")

if "article" not in st.session_state:
    st.session_state.article = ""


def use_example():
    choice = st.session_state.example
    if choice in EXAMPLE_NAMES:
        st.session_state.article = EXAMPLES[EXAMPLE_NAMES.index(choice)]["text"]


st.selectbox("Try an example", [NO_EXAMPLE] + EXAMPLE_NAMES, key="example", on_change=use_example)
text = st.text_area("Kinyarwanda news article", key="article", height=260,
                    placeholder="Andika cyangwa ushyiremo inkuru hano… (paste an article here)")

if st.button("Classify", type="primary"):
    if len(text.split()) < 5:
        st.warning("Please paste a longer Kinyarwanda news text (at least one or two sentences).")
    else:
        with st.spinner("Classifying…"):
            ranked, n_tokens = classify(text, tokenizer, model, id2label)
        label, conf = ranked[0]
        st.subheader(f"Predicted category: {label}")
        st.caption(f"Confidence: {conf:.1%}")
        if conf < LOW_CONFIDENCE:
            st.warning("Low confidence. The text may cover several topics, belong to none of the 14 "
                       "categories, or not be in Kinyarwanda. Treat this prediction with caution.")
        if n_tokens > MAX_LEN - 2:
            st.info(f"The text is {n_tokens} tokens long; the model only reads the first {MAX_LEN - 2}.")
        st.markdown("**Top-5 categories**")
        for name, p in ranked[:5]:
            st.progress(min(max(p, 0.0), 1.0), text=f"{name}: {p:.1%}")

with st.expander("About this model"):
    st.markdown(f"""
- **Model:** `{INFO['base_model']}` fine-tuned on the KINNEWS corpus (Niyongabo et al., 2020): experiment {INFO['exp_id']}, max {MAX_LEN} tokens, class-weighted loss. Hosted on the Hugging Face Hub: `{INFO['model_repo']}`.
- **How the app works:** the model is downloaded once and cached; your text is tokenised, passed through the model, and the softmax probabilities are shown.
- **Test results (deduplicated KINNEWS test set, {INFO['test_size']} articles):** accuracy {INFO['test_accuracy']:.1%}, macro-F1 {INFO['test_macro_f1']:.3f}.
- **Limitations:** trained on Rwandan online news from about 2019–2020; rare categories (tourism, fashion, environment, education) are much less reliable; the model can partly rely on source names and bylines; non-news or non-Kinyarwanda text gives unreliable outputs.
- **Code and report:** {INFO['github_url']}
""")

Writing /content/app/streamlit_app.py


## Step 7: `requirements.txt`
`transformers` is pinned to the version used for training. The extra index installs the smaller **CPU-only** build of PyTorch.

In [ ]:
import transformers
open(f'{APP_DIR}/requirements.txt', 'w').write(
    '--extra-index-url https://download.pytorch.org/whl/cpu\n'
    f'transformers=={transformers.__version__}\n'
    'torch\nsentencepiece\nprotobuf\nstreamlit\n')
print(open(f'{APP_DIR}/requirements.txt').read())

--extra-index-url https://download.pytorch.org/whl/cpu
transformers==5.18.0
torch
sentencepiece
protobuf
streamlit



## Step 8: Test the prediction code (loads the model from the Hub, exactly like the deployed app)

In [ ]:
import sys, importlib
sys.path.insert(0, APP_DIR)
import predictor; importlib.reload(predictor)
tok, mdl, id2label = predictor.load_model()
for e in examples:
    ranked, n_tok = predictor.classify(e['text'], tok, mdl, id2label)
    top = ', '.join(f'{c} {p:.0%}' for c, p in ranked[:3])
    print(f"[{e['description']}]\n   -> {top}  ({n_tok} tokens)\n")

config.json:   0%|          | 0.00/1.41k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/426 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.98M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  446MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/137 [00:00<?, ?it/s]

[sport article]
   -> sport 99%, entertainment 0%, economy 0%  (219 tokens)

[health article]
   -> health 98%, politic 1%, history 0%  (237 tokens)

[religion article]
   -> religion 99%, entertainment 0%, fashion 0%  (210 tokens)

[economy article]
   -> economy 99%, politic 0%, health 0%  (237 tokens)

[labelled entertainment (model error on full article: sport)]
   -> sport 99%, entertainment 0%, economy 0%  (225 tokens)

[labelled entertainment (model error on full article: sport)]
   -> sport 99%, entertainment 0%, economy 0%  (203 tokens)

[English text (out of scope)]
   -> economy 95%, environment 2%, politic 1%  (37 tokens)



## Step 9:  `app`

In [ ]:
dst = f'{PROJECT_DIR}/app'
if os.path.exists(dst): shutil.rmtree(dst)
shutil.copytree(APP_DIR, dst, ignore=shutil.ignore_patterns('__pycache__'))
print(sorted(os.listdir(dst)))

['examples.json', 'model_info.json', 'predictor.py', 'requirements.txt', 'streamlit_app.py']
